# EfficientNet-Lite0 Stage 3 — D3 RealWaste External Inference Sustainability (Frozen Classification Checkpoints)

This corrected notebook follows the actual **EfficientNet-Lite0(3).ipynb** Stage 2 sustainability record and the standalone EfficientNet-Lite0 frozen Stage 3 classification notebook for the model checkpoint structure. It does **not** train or adapt any model.

- **Stage 2 historical monitoring:** retained as reported, including its original CodeCarbon emissions. The original Stage 2 notebook monitored training with epoch-wise evaluation; hence the values should be described as *historical training-session measurements*, not isolated backpropagation cost.
- **Stage 3 D2 inference:** 3,497 independently screened external six-class images, all five frozen standalone classification checkpoints, 32 × 32 RGB, batch 32, two or more complete inference passes, warm-up excluded, 15 s CodeCarbon sampling, GPU synchronization, no fine-tuning.
- **Carbon accounting:** preserve original CodeCarbon emissions.csv and report standardized CO₂e separately as total monitored kWh × **0.475 kg CO₂e/kWh**. Historical Stage 2 reported means are preserved at source precision. Five-seed sample SD uses ddof=1. Display ordinary decimals (not scientific notation).
- **Comparison:** keep historical Stage 2 session and new Stage 3 inference results separate. Only compare per-image efficiency among standalone models after verifying identical hardware, inputs, checkpoint hashes, batch size, and monitoring boundaries.

The notebook stops if input/checkpoint structure or manifest verification fails. It does not silently replace checkpoints.

**D3-only frozen evaluation.** This adapts the original sustainability workflow (CodeCarbon, five seeds, fixed carbon intensity, historical Stage 2 reference and ZIP) to the exact class-file paths and locked 3,497-image RealWaste manifest. Original sustainability code executes a 384×384 resize; reporting is corrected to reflect actual execution. It does NOT fine-tune or select checkpoints on D3.

**Preprocessing comparability:** 384×384 comes from the executable sustainability notebook, while the classification D3 notebook uses 184×184. To reproduce classification accuracy exactly, change `INPUT_RESOLUTION` to `184` before the first evaluation, and describe that change in the paper. The checkpoint and D3 paths remain identical in either case.

## 1. Fixed configuration and input files

In [1]:
from __future__ import annotations
import os, sys, json, hashlib, zipfile, subprocess, importlib.util, traceback, re
from pathlib import Path, PurePosixPath
from collections import Counter
from datetime import datetime, timezone

# Kaggle input location confirmed by the user; adjust ONLY if Kaggle moves the dataset.
INPUT_ROOT = Path('/kaggle/input/datasets/akash264208/manifist-part2/Section3')
OUTPUT = Path('/kaggle/working/EfficientNet_Lite0_D3_Inference_Sustainability')
OUTPUT.mkdir(parents=True, exist_ok=True)
SEEDS = (7, 49, 112, 345, 476)
MODEL_NAME = 'EfficientNet-Lite0'
CARBON_FACTOR = 0.475
MIN_COMPLETE_PASSES = 2
MIN_MONITOR_SECONDS = 35.0
CODECARBON_SAMPLING_SECONDS = 15
ZIP_PATH = Path('/kaggle/working/EfficientNet_Lite0_D3_Inference_Sustainability_All_Outputs.zip')
CHECKPOINT_MODEL_METADATA = 'EfficientNet-Lite0'
CHECKPOINT_ARCHITECTURE = 'timm.efficientnet_lite0'
CLASSES = ('cardboard','glass','metal','paper','plastic','trash')
CLASS_TO_ID = {name:i for i,name in enumerate(CLASSES)}
EXPECTED_EXTERNAL_COUNT = 3497
EXPECTED_EXTERNAL_MANIFEST_FINGERPRINT = '0350db99b6be93bfaa9a1dfcdaf99d9c127404658a6222c722bc83fb131cf0e0'
# Set this to the exact Kaggle folder containing the D3 Section 2 CSV files.
# Example: '/kaggle/input/datasets/YOUR_ACCOUNT/YOUR_D3_SECTION2_OUTPUTS'
# If None, the notebook locates unique matching files under /kaggle/input.
D3_MANIFEST_ROOT = '/kaggle/input/datasets/akash264208/manifist-part2/Section3/manifist'
BATCH_SIZE = 32

# Optional genuine historical Stage 2 standalone EfficientNet-Lite0 prediction files.
HISTORICAL_REFERENCE_DIR = None
REQUIRE_HISTORICAL_REFERENCE = False  # Set True to STOP external evaluation if references are absent.

assert INPUT_ROOT.is_dir(), f'Input dataset was not attached: {INPUT_ROOT}'
print('Kaggle input:', INPUT_ROOT)
print('Output directory:', OUTPUT)
print('Seeds:', SEEDS)
print('Frozen standalone EfficientNet-Lite0 model; no training or fine-tuning.')
# Preserve the original sustainability notebook's executable 384x384 input preprocessing.
INPUT_RESOLUTION = 384


Kaggle input: /kaggle/input/datasets/akash264208/manifist-part2/Section3
Output directory: /kaggle/working/EfficientNet_Lite0_D3_Inference_Sustainability
Seeds: (7, 49, 112, 345, 476)
Frozen standalone EfficientNet-Lite0 model; no training or fine-tuning.


## 2. Dependencies and strict classification-file checkpoint inventory

In [2]:
# Exact CodeCarbon version and monitoring implementation used in companion Stage 3 notebooks.
try:
    from importlib.metadata import version as installed_version
    _cc_version=installed_version('codecarbon')
except Exception:
    _cc_version=None
if _cc_version is None:
    subprocess.check_call([sys.executable,'-m','pip','install','-q','codecarbon==3.3.2'])
from codecarbon import EmissionsTracker
import time, shutil, platform
print('CodeCarbon version:',installed_version('codecarbon'))

if importlib.util.find_spec('kagglehub') is None:
    subprocess.check_call([sys.executable,'-m','pip','install','-q','kagglehub'])
import kagglehub
if importlib.util.find_spec('timm') is None:
    subprocess.check_call([sys.executable,'-m','pip','install','-q','timm'])
import timm
import numpy as np
import pandas as pd
import torch
from torch import nn
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
import matplotlib.pyplot as plt

DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type=='cuda':
    torch.backends.cudnn.deterministic=True
    torch.backends.cudnn.benchmark=False
    torch.backends.cuda.matmul.allow_tf32=False
print('Device:',DEVICE,'| torch:',torch.__version__,'| timm:',timm.__version__)


def only_directory_with_name(root, name):
    matches = sorted(p for p in root.rglob('*') if p.is_dir() and p.name.casefold() == name.casefold())
    if len(matches) != 1:
        raise RuntimeError(f'Expected exactly one directory {name!r} under {root}; found {matches}')
    return matches[0]

def unique_by_filename(root, filename):
    matches = sorted(p for p in root.rglob(filename) if p.is_file())
    if len(matches) != 1:
        raise RuntimeError(f'Expected exactly one {filename!r} in {root}: {matches}')
    return matches[0]

CKPT_ROOT = only_directory_with_name(INPUT_ROOT, 'Classification_checkpoints')
REMNET_CKPT_DIR = only_directory_with_name(CKPT_ROOT, 'Remnet')
RESULTS_ROOT = only_directory_with_name(INPUT_ROOT, 'per_seed_performance_summary')

def d3_csv_path(filename):
    root = Path(D3_MANIFEST_ROOT) if D3_MANIFEST_ROOT else Path('/kaggle/input')
    if not root.is_dir():
        raise FileNotFoundError(f'D3 manifest folder not attached: {root}')
    return unique_by_filename(root, filename)

EXTERNAL_MANIFEST = d3_csv_path('D3_Section2_Final_Automated_Screened_Six_Class_Manifest.csv')
CLASS_MAPPING = d3_csv_path('D3_Section2_Class_Mapping.csv')

# Match only dedicated standalone EfficientNet-Lite0 directories, not the REMNET ensemble tree.
standalone_dirs=[p for p in CKPT_ROOT.iterdir() if p.is_dir()
                 and 'efficient' in re.sub(r'[^a-z0-9]','',p.name.lower())
                 and 'remnet' not in re.sub(r'[^a-z0-9]','',p.name.lower())]
if len(standalone_dirs)!=1:
    raise RuntimeError('Expected exactly one standalone EfficientNet-Lite0 directory under '
                       f'{CKPT_ROOT}; found {standalone_dirs}. Inspect Kaggle attachments.')
STANDALONE_CKPT_DIR=standalone_dirs[0]
print('Standalone EfficientNet-Lite0 checkpoint directory:',STANDALONE_CKPT_DIR)

CHECKPOINTS={}
for seed in SEEDS:
    matches=sorted(p for p in STANDALONE_CKPT_DIR.rglob('*.pt') if
                   re.search(rf'(?:^|[^0-9])seed[_-]?{seed}(?:[^0-9]|$)',p.stem,re.IGNORECASE))
    if len(matches)!=1:
        raise RuntimeError(f'Expected precisely one standalone EfficientNet-Lite0 .pt checkpoint for seed {seed}: {matches}')
    CHECKPOINTS[seed]=matches[0]

all_pts = sorted(CKPT_ROOT.rglob('*.pt'))
standalone_pts = sorted(STANDALONE_CKPT_DIR.rglob('*.pt'))
remnet_pts = sorted(REMNET_CKPT_DIR.rglob('*.pt'))
assert len(standalone_pts) == 5, f'Expected 5 standalone EfficientNet-Lite0 checkpoints, found {len(standalone_pts)}'
assert len(remnet_pts) == 15, f'Expected 15 REMNET ensemble constituent checkpoints, found {len(remnet_pts)}'
assert len(all_pts) == 30, f'Expected 30 total Stage 2 checkpoints, found {len(all_pts)}'
assert not set(CHECKPOINTS.values()).intersection(remnet_pts)

HISTORICAL_PER_SEED_FILES=sorted(p for p in RESULTS_ROOT.rglob('*.csv') if 'efficient' in p.name.lower() and 'per_seed' in p.name.lower())
HISTORICAL_SUMMARY_FILES=sorted(p for p in RESULTS_ROOT.rglob('*.csv') if 'efficient' in p.name.lower() and ('summary' in p.name.lower() or '5_seed' in p.name.lower()))
HISTORICAL_PER_SEED=HISTORICAL_PER_SEED_FILES[0] if len(HISTORICAL_PER_SEED_FILES)==1 else None
HISTORICAL_SUMMARY=HISTORICAL_SUMMARY_FILES[0] if len(HISTORICAL_SUMMARY_FILES)==1 else None
print('Total .pt checkpoints:', len(all_pts), '| Standalone EfficientNet-Lite0:', len(standalone_pts),
      '| REMNET constituent checkpoints:', len(remnet_pts))
for seed, path in CHECKPOINTS.items(): print('STANDALONE EfficientNet-Lite0', seed, ':', path)
for path in [EXTERNAL_MANIFEST, CLASS_MAPPING]: print('CSV:', path)
print('Historical EfficientNet-Lite0 per-seed:', HISTORICAL_PER_SEED or 'not present')
print('Historical EfficientNet-Lite0 summary:', HISTORICAL_SUMMARY or 'not present')

CodeCarbon version: 3.3.2
Device: cuda | torch: 2.11.0+cu130 | timm: 1.0.30
Standalone EfficientNet-Lite0 checkpoint directory: /kaggle/input/datasets/akash264208/manifist-part2/Section3/Classification_checkpoints/efficientnet_lite0
Total .pt checkpoints: 30 | Standalone EfficientNet-Lite0: 5 | REMNET constituent checkpoints: 15
STANDALONE EfficientNet-Lite0 7 : /kaggle/input/datasets/akash264208/manifist-part2/Section3/Classification_checkpoints/efficientnet_lite0/EfficientNet_Lite0_timm_seed7.pt
STANDALONE EfficientNet-Lite0 49 : /kaggle/input/datasets/akash264208/manifist-part2/Section3/Classification_checkpoints/efficientnet_lite0/EfficientNet_Lite0_timm_seed49.pt
STANDALONE EfficientNet-Lite0 112 : /kaggle/input/datasets/akash264208/manifist-part2/Section3/Classification_checkpoints/efficientnet_lite0/EfficientNet_Lite0_timm_seed112.pt
STANDALONE EfficientNet-Lite0 345 : /kaggle/input/datasets/akash264208/manifist-part2/Section3/Classification_checkpoints/efficientnet_lite0/Effici

## 3. Verify the locked RealWaste D3 manifest and six-class mapping

In [3]:
def file_sha256(path, chunk_size=1024*1024):
    h = hashlib.sha256()
    with open(path,'rb') as stream:
        for b in iter(lambda:stream.read(chunk_size),b''):
            h.update(b)
    return h.hexdigest()

def canonical_external_fingerprint(df):
    h = hashlib.sha256()
    for row in df.sort_values('external_relative_path').itertuples(index=False):
        h.update((row.external_relative_path+'|'+str(int(row.output_index))+'|'+
                  row.sha256+'\n').encode('utf-8'))
    return h.hexdigest()

external_df = pd.read_csv(EXTERNAL_MANIFEST, dtype=str, keep_default_na=False)
mapping_df = pd.read_csv(CLASS_MAPPING, dtype=str, keep_default_na=False)

assert set(['external_relative_path','class','output_index','sha256','final_status']) <= set(external_df.columns)
assert set(['d3_source_label','remnet_trained_class','remnet_output_index','include_for_primary_d3']) <= set(mapping_df.columns)
assert len(external_df)==EXPECTED_EXTERNAL_COUNT, 'Unexpected D3 manifest count (should be 3497).'
assert external_df['sha256'].nunique()==EXPECTED_EXTERNAL_COUNT, 'Duplicate hashes in external manifest.'
assert external_df['external_relative_path'].nunique()==EXPECTED_EXTERNAL_COUNT
assert set(external_df['class']) == set(CLASSES)
assert external_df['final_status'].eq('ELIGIBLE_AFTER_REFERENCE_VERIFICATION').all(), 'Noneligible images in frozen manifest.'
included_mapping = mapping_df.loc[mapping_df['include_for_primary_d3'].str.lower().eq('true')].copy()
assert len(included_mapping) == 6
assert set(included_mapping['remnet_trained_class']) == set(CLASSES)
for item in included_mapping.itertuples(index=False):
    assert int(item.remnet_output_index) == CLASS_TO_ID[item.remnet_trained_class]
    if 'class_review' in mapping_df.columns:
        assert item.class_review == 'CONFIRMED_BY_USER'
assert set(external_df['source_category']) == set(included_mapping['d3_source_label'])
assert all(int(i)==CLASS_TO_ID[c] for c,i in zip(external_df['class'],external_df['output_index']))

EXTERNAL_FINGERPRINT=canonical_external_fingerprint(external_df)
assert EXTERNAL_FINGERPRINT == EXPECTED_EXTERNAL_MANIFEST_FINGERPRINT, (EXTERNAL_FINGERPRINT,'External cohort fingerprint mismatch')
print('D3 frozen manifest verified:',len(external_df),'Fingerprint:',EXTERNAL_FINGERPRINT)
print('Six class mapping:',CLASS_TO_ID)
print('External per-class counts:',external_df['class'].value_counts().sort_index().to_dict())

D3 frozen manifest verified: 3497 Fingerprint: 0350db99b6be93bfaa9a1dfcdaf99d9c127404658a6222c722bc83fb131cf0e0
Six class mapping: {'cardboard': 0, 'glass': 1, 'metal': 2, 'paper': 3, 'plastic': 4, 'trash': 5}
External per-class counts: {'cardboard': 454, 'glass': 419, 'metal': 757, 'paper': 500, 'plastic': 889, 'trash': 478}


## 4. Resolve the exact RealWaste dataset path from the classification notebook

In [4]:
# Stage 3 D3 dataset path — fixed mounted Kaggle input.
EXTERNAL_HANDLE = 'akash264208/real-waste'
external_download = Path('/kaggle/input/datasets/akash264208/real-waste')
external_download_method = 'attached Kaggle dataset'
assert external_download.is_dir(), f'D3 RealWaste dataset missing: {external_download}'
print('D3 RealWaste:', external_download)

D3 RealWaste: /kaggle/input/datasets/akash264208/real-waste


## 5. SHA-256 verify all 3,497 D3 RealWaste images (unchanged class-file manifest)

In [5]:
IMG_EXT={'.jpg','.jpeg','.png','.webp','.bmp'}

def safe_parts(relative):
    p=PurePosixPath(str(relative).replace('\\','/'))
    if p.is_absolute() or not p.parts or any(s in ('','.','..') for s in p.parts):
        raise ValueError(f'Unsafe image relative path: {relative!r}')
    return p.parts

def external_class_roots(root):
    expected=set(included_mapping['d3_source_label'])
    candidates=[root] + sorted((p for p in root.rglob('*') if p.is_dir()), key=str)
    return [p for p in candidates if all((p/c).is_dir() for c in expected)]

def resolve_external(external,download):
    candidates=external_class_roots(download)
    print('D3 RealWaste class-root candidates:',candidates)
    if not candidates: raise FileNotFoundError('Missing RealWaste source-category directories.')
    expected_rows=[(safe_parts(r.external_relative_path),r.sha256,r.source_category)
                   for _,r in external.iterrows()]
    for root in candidates:
        try:
            dirs=[p for p in root.iterdir() if p.is_dir()]
            image_counts={p.name:sum(x.is_file() and x.suffix.lower() in IMG_EXT
                         for x in p.rglob('*')) for p in dirs}
            if len(image_counts)!=9 or sum(image_counts.values())!=4752:
                raise RuntimeError(f'Expected nine RealWaste folders / 4752 images: {image_counts}')
            if sum(image_counts[c] for c in included_mapping['d3_source_label'])!=3587:
                raise RuntimeError('D3 six-category source total differs from Section 2.')
            resolved=[]
            for parts,digest,source_cls in expected_rows:
                image=root.joinpath(*parts)
                if not image.is_file() or parts[0]!=source_cls:
                    raise FileNotFoundError(str(image))
                if file_sha256(image)!=digest:
                    raise ValueError(f'D3 image hash mismatch: {image}')
                resolved.append(str(image))
            print('D3 RealWaste images SHA-256 verified:',len(resolved),'root:',root)
            return resolved,root,image_counts
        except (FileNotFoundError,ValueError,RuntimeError) as exc:
            print('D3 candidate rejected:',root,'reason:',str(exc)[:180])
    raise RuntimeError('No RealWaste root matches the 3497 locked image hashes.')

external_paths, external_image_root, d3_counts=resolve_external(external_df,external_download)
external_df=external_df.copy(); external_df['resolved_path']=external_paths
external_eval_df=external_df.sort_values('external_relative_path').reset_index(drop=True)
assert len(external_eval_df)==3497
print('External evaluated images:',len(external_eval_df))

D3 RealWaste class-root candidates: [PosixPath('/kaggle/input/datasets/akash264208/real-waste/realwaste-main/RealWaste')]
D3 RealWaste images SHA-256 verified: 3497 root: /kaggle/input/datasets/akash264208/real-waste/realwaste-main/RealWaste
External evaluated images: 3497


## 6. Frozen EfficientNet-Lite0 checkpoint restore and sustainability inference preprocessing

In [6]:
TEST_TRANSFORM=transforms.Compose([
    transforms.Resize((INPUT_RESOLUTION,INPUT_RESOLUTION)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5)),
])

# Exact architecture is identified from the ORIGINAL standalone classification checkpoint.
# Stage 2 training used timm.efficientnet_lite0 inside a self.model wrapper.
# The archived classification file may store either the timm bare keys or wrapper keys.
# No state tensor is modified or omitted; weights_only loading and strict=True are mandatory.
class EfficientNetLiteModel(nn.Module):
    def __init__(self,num_classes):
        super().__init__()
        self.model=timm.create_model('efficientnet_lite0',pretrained=False,num_classes=num_classes)
    def forward(self,x):
        return self.model(x)

def _same_names_and_shapes(model_state,ckpt_state):
    return set(model_state)==set(ckpt_state) and all(
        tuple(model_state[k].shape)==tuple(ckpt_state[k].shape) for k in model_state)

def _choose_exact_efficientnet_structure(state):
    variants={'unaltered':state}
    if all(k.startswith('module.') for k in state):
        variants['remove_module.']={k[7:]:v for k,v in state.items()}
    if all(k.startswith('_orig_mod.') for k in state):
        variants['remove__orig_mod.']={k[len('_orig_mod.'):]:v for k,v in state.items()}
    matches=[]
    for name,constructor in (
       ('timm.efficientnet_lite0',lambda:timm.create_model('efficientnet_lite0',pretrained=False,num_classes=6)),
       ('wrapped.timm.efficientnet_lite0',lambda:EfficientNetLiteModel(6)),
    ):
        model=constructor()
        for variant,adjusted in variants.items():
            additional={'unaltered':adjusted}
            if name.startswith('wrapped.') and not all(k.startswith('model.') for k in adjusted):
                additional['add_model.']={'model.'+k:v for k,v in adjusted.items()}
            if not name.startswith('wrapped.') and all(k.startswith('model.') for k in adjusted):
                additional['remove_model.']={k[len('model.'):]:v for k,v in adjusted.items()}
            for prefix_conversion,converted in additional.items():
                if _same_names_and_shapes(model.state_dict(),converted):
                    matches.append((name,variant+' / '+prefix_conversion,model,converted))
                    break
            if any(x[0]==name for x in matches):break
        if not any(x[0]==name for x in matches):del model
    if not matches:
        raise RuntimeError('No exact timm EfficientNet-Lite0 6-class match. '
                           'Checkpoint keys start with: '+str(list(state)[:10])+
                           '. Verify original classification checkpoint, do not disable strict loading.')
    # Prefer the bare timm model for bare checkpoints; wrapped model otherwise.
    preferred='wrapped.timm.efficientnet_lite0' if all(k.startswith('model.') for k in state) else 'timm.efficientnet_lite0'
    chosen=next((item for item in matches if item[0]==preferred),matches[0])
    return chosen

class LockedImageDataset(Dataset):
    def __init__(self,df,kind):
        self.df=df.reset_index(drop=True)
        self.kind=kind
        self.path_col='external_relative_path'
    def __len__(self):return len(self.df)
    def __getitem__(self,i):
        row=self.df.iloc[i]
        with Image.open(row['resolved_path']) as im:
            x=TEST_TRANSFORM(im.convert('RGB'))
        return x,int(CLASS_TO_ID[row['class']]),i

def locked_loader(df,kind):
    return DataLoader(LockedImageDataset(df,kind),batch_size=BATCH_SIZE,
                      shuffle=False,num_workers=0,pin_memory=DEVICE.type=='cuda')

def restore_checkpoint(seed):
    p=CHECKPOINTS[seed]  # standalone classification checkpoints ONLY
    ckpt=torch.load(p,map_location='cpu',weights_only=True)
    if not isinstance(ckpt,dict):raise RuntimeError(f'Unexpected checkpoint container: {p}')
    state_key=next((key for key in ('state_dict','model_state_dict') if key in ckpt),None)
    state=ckpt[state_key] if state_key is not None else ckpt
    if state_key is not None:
        if 'seed' in ckpt and int(ckpt['seed'])!=seed:
            raise RuntimeError(f'Checkpoint seed mismatch: {p}')
        if 'model' in ckpt and 'efficientnet' not in re.sub(r'[^a-z0-9]','',str(ckpt['model']).lower()):
            raise RuntimeError(f'Checkpoint model mismatch: {p}')
        if 'architecture' in ckpt and str(ckpt['architecture']).lower() not in ('timm.efficientnet_lite0','efficientnet_lite0'):
            raise RuntimeError(f'Unexpected declared architecture: {ckpt["architecture"]}')
        if 'num_classes' in ckpt and int(ckpt['num_classes'])!=6:
            raise RuntimeError(f'Checkpoint does not have six classes: {p}')
    if not isinstance(state,dict) or not state or not all(torch.is_tensor(v) for v in state.values()):
        raise RuntimeError(f'Checkpoint does not contain a tensor-only state_dict: {p}')
    architecture,prefix_conversion,model,adjusted=_choose_exact_efficientnet_structure(state)
    model.load_state_dict(adjusted,strict=True)
    model.to(DEVICE).eval()
    for param in model.parameters():param.requires_grad_(False)
    if not hasattr(restore_checkpoint,'detected_architectures'):
        restore_checkpoint.detected_architectures={}
    restore_checkpoint.detected_architectures[seed]={
        'framework':'timm','architecture':architecture,'prefix_conversion':prefix_conversion}
    print(f'Validated EfficientNet-Lite0 seed {seed}: {architecture} / {prefix_conversion} / strict=True')
    return model,ckpt,file_sha256(p)

def infer(model,df,kind):
    labels=[]; probs=[]; positions=[]
    with torch.inference_mode():
        for images,y,index in locked_loader(df,kind):
            pred=torch.softmax(model(images.to(DEVICE,non_blocking=True)),dim=1)
            probs.append(pred.cpu().numpy().astype(np.float64))
            labels.append(y.numpy());positions.append(index.numpy())
    labels=np.concatenate(labels).astype(np.int64)
    probs=np.concatenate(probs)
    positions=np.concatenate(positions)
    assert np.array_equal(positions,np.arange(len(df))), 'Inference image order changed'
    assert np.array_equal(labels,df['class'].map(CLASS_TO_ID).to_numpy()), 'Class-index mismatch'
    assert probs.shape==(len(df),6) and np.isfinite(probs).all(), 'Invalid probability array'
    assert np.allclose(probs.sum(axis=1),1.0,atol=2e-6), 'Probabilities do not sum to 1'
    return labels,probs

_smoke,_smoke_info,_smoke_digest=restore_checkpoint(SEEDS[0])
with torch.inference_mode():
    _test_logits=_smoke(torch.zeros(1,3,INPUT_RESOLUTION,INPUT_RESOLUTION,device=DEVICE))
assert tuple(_test_logits.shape)==(1,6)
print('EfficientNet-Lite0 strict-load and six-class smoke test: PASS')
del _smoke,_smoke_info,_test_logits
if DEVICE.type=='cuda':torch.cuda.empty_cache()

Validated EfficientNet-Lite0 seed 7: timm.efficientnet_lite0 / unaltered / unaltered / strict=True
EfficientNet-Lite0 strict-load and six-class smoke test: PASS


## 7. CodeCarbon inference monitoring (original sustainability protocol)

In [7]:
LOG_ROOT = OUTPUT/'CodeCarbon_Logs'
LOG_ROOT.mkdir(parents=True,exist_ok=True)

def make_tracker(log_dir, seed):
    log_dir.mkdir(parents=True,exist_ok=True)
    prior = log_dir/'emissions.csv'
    if prior.exists():
        prior.rename(log_dir/f'emissions_previous_{time.time_ns()}.csv')
    options = dict(
        project_name=f'EfficientNetLite0_Stage3_seed{seed}_External3497',
        output_dir=str(log_dir), output_file='emissions.csv',
        save_to_file=True, tracking_mode='machine',
        measure_power_secs=CODECARBON_SAMPLING_SECONDS,
        pue=1.0, log_level='error'
    )
    if DEVICE.type=='cuda': options['gpu_ids']=[0]
    return EmissionsTracker(**options)

def gpu_sync():
    if DEVICE.type=='cuda': torch.cuda.synchronize()

def measured_row(log_dir):
    filepath=log_dir/'emissions.csv'
    if not filepath.is_file():
        raise FileNotFoundError(f'CodeCarbon emissions file missing: {filepath}')
    df=pd.read_csv(filepath)
    if df.empty: raise RuntimeError(f'Empty CodeCarbon file: {filepath}')
    row=df.iloc[-1].to_dict()
    energy=float(row['energy_consumed'])
    if not np.isfinite(energy) or energy<0: raise RuntimeError(f'Invalid CodeCarbon energy {energy}')
    emissions=float(row['emissions']) if 'emissions' in row and pd.notna(row['emissions']) else np.nan
    return row,energy,emissions

def monitored_efficientnetlite0_inference(seed, model):
    loader=locked_loader(external_eval_df,'external')
    model.eval()
    # Two identical warm-up forward-only batches, before monitoring starts.
    images, _, _ = next(iter(loader))
    images=images.to(DEVICE,non_blocking=True)
    with torch.inference_mode():
        for _ in range(2):
            p=torch.softmax(model(images),dim=1)
            if p.shape[1]!=6: raise RuntimeError('Unexpected six-class output during warm-up')
    gpu_sync()
    del images,p
    log_dir=LOG_ROOT/f'EfficientNetLite0_seed{seed}'/'inference'
    tracker=make_tracker(log_dir,seed)
    passes=0
    processed=0
    labels_first=[]; pred_first=[]
    started=False
    try:
        tracker.start()
        started=True
        gpu_sync()
        t0=time.perf_counter()
        with torch.inference_mode():
            while passes<MIN_COMPLETE_PASSES or (time.perf_counter()-t0)<MIN_MONITOR_SECONDS:
                current_positions=[]
                for images,y,positions in loader:
                    images=images.to(DEVICE,non_blocking=True)
                    probs=torch.softmax(model(images),dim=1)
                    if tuple(probs.shape)!=(len(y),6):
                        raise RuntimeError('Six-class output mismatch')
                    if passes==0:
                        labels_first.extend(y.tolist())
                        pred_first.extend(probs.argmax(dim=1).cpu().tolist())
                        current_positions.extend(positions.tolist())
                    processed+=len(y)
                if passes==0 and current_positions!=list(range(len(external_eval_df))):
                    raise RuntimeError('External image order changed during monitored inference')
                passes+=1
        gpu_sync()
        seconds=time.perf_counter()-t0
    finally:
        if started:
            gpu_sync()
            tracker.stop()
    if processed!=passes*len(external_eval_df):
        raise RuntimeError(f'Incomplete inference pass: {processed} images / {passes} passes')
    row,energy,cc_emissions=measured_row(log_dir)
    if seconds<=0 or processed<=0:raise RuntimeError('Invalid measured workload')
    ytrue=np.asarray(labels_first,dtype=int)
    ypred=np.asarray(pred_first,dtype=int)
    expected=external_eval_df['class'].map(CLASS_TO_ID).to_numpy()
    assert np.array_equal(ytrue,expected)
    accuracy_pct=100*float((ytrue==ypred).mean())
    macro_p, macro_r, macro_f1, _=precision_recall_fscore_support(
        ytrue,ypred,labels=list(range(6)),average='macro',zero_division=0)
    macro_p_pct,macro_r_pct,macro_f1_pct=100*macro_p,100*macro_r,100*macro_f1
    return {
       'model':'EfficientNet-Lite0','seed':int(seed),
       'unique_external_images':len(external_eval_df),
       'number_complete_repetitions':passes,'total_processed_images':processed,
       'batch_size':BATCH_SIZE,'input_resolution':f'{INPUT_RESOLUTION}x{INPUT_RESOLUTION}',
       'monitoring_seconds':seconds,'total_inference_energy_kWh':energy,
       'codecarbon_emissions_kgCO2e':cc_emissions,
       'standardized_CO2e_kg_fixed_factor':energy*CARBON_FACTOR,
       'standardized_CO2_factor_kg_per_kWh':CARBON_FACTOR,
       'energy_kWh_per_processed_image':energy/processed,
       'time_seconds_per_processed_image':seconds/processed,
       'throughput_images_per_second':processed/seconds,
       'first_pass_accuracy_pct_diagnostic':accuracy_pct,
       'first_pass_macro_precision_pct':macro_p_pct,
       'first_pass_macro_recall_pct':macro_r_pct,
       'first_pass_macro_f1_pct':macro_f1_pct,
       'codecarbon_duration_seconds':row.get('duration',np.nan),
       'codecarbon_energy_column':'energy_consumed',
       'codecarbon_version':installed_version('codecarbon'),
       'gpu':torch.cuda.get_device_name(0) if DEVICE.type=='cuda' else 'CPU',
       'raw_emissions_csv':str(log_dir/'emissions.csv')
    }


## 8. Monitor five frozen EfficientNet-Lite0 classification checkpoints on all 3,497 D3 images

In [8]:
sustainability_rows=[]
checkpoint_rows=[]
for seed in SEEDS:
    print(f'\n=== STANDALONE EFFICIENTNET-LITE0 STAGE 3 EXTERNAL INFERENCE — SEED {seed} ===',flush=True)
    model,meta,digest=restore_checkpoint(seed)
    checkpoint_rows.append({
        'model':'EfficientNet-Lite0','seed':seed,'checkpoint_path':str(CHECKPOINTS[seed]),
        'checkpoint_file_sha256':digest,'checkpoint_seed':meta.get('seed'),
        'checkpoint_model':meta.get('model'),'checkpoint_architecture_metadata':meta.get('architecture'),
        'split_sha256':meta.get('split_sha256'),
        'classification_checkpoint_source':'standalone Classification_checkpoints/EfficientNet-Lite0/*.pt',
        'preprocessing':f'RGB Resize({INPUT_RESOLUTION},{INPUT_RESOLUTION}) ToTensor Normalize(.5,.5,.5)',
        'checkpoint_loading':'strict=True; full key/shape match, optional exact wrapper-prefix adjustment'})
    result=monitored_efficientnetlite0_inference(seed,model)
    sustainability_rows.append(result)
    print(f"Seed {seed} completed | external: {result['unique_external_images']} images × "
          f"{result['number_complete_repetitions']} passes = {result['total_processed_images']} processed "
          f"| energy {result['total_inference_energy_kWh']:.9f} kWh "
          f"| original CodeCarbon CO2e {result['codecarbon_emissions_kgCO2e']:.9f} kg "
          f"| standardized CO2e {result['standardized_CO2e_kg_fixed_factor']:.9f} kg "
          f"| monitored runtime {result['monitoring_seconds']:.3f} s",flush=True)
    del model
    if DEVICE.type=='cuda': torch.cuda.empty_cache()
checkpoint_df=pd.DataFrame(checkpoint_rows)
checkpoint_df.to_csv(OUTPUT/'EfficientNet_Lite0_Stage3_Classification_Checkpoint_Registry.csv',index=False)



=== STANDALONE EFFICIENTNET-LITE0 STAGE 3 EXTERNAL INFERENCE — SEED 7 ===
Validated EfficientNet-Lite0 seed 7: timm.efficientnet_lite0 / unaltered / unaltered / strict=True


[codecarbon WARNING @ 16:50:58] Multiple instances of codecarbon are allowed to run at the same time.


Seed 7 completed | external: 3497 images × 2 passes = 6994 processed | energy 0.001395178 kWh | original CodeCarbon CO2e 0.000401321 kg | standardized CO2e 0.000662710 kg | monitored runtime 76.798 s

=== STANDALONE EFFICIENTNET-LITE0 STAGE 3 EXTERNAL INFERENCE — SEED 49 ===
Validated EfficientNet-Lite0 seed 49: timm.efficientnet_lite0 / unaltered / unaltered / strict=True
Seed 49 completed | external: 3497 images × 2 passes = 6994 processed | energy 0.001430103 kWh | original CodeCarbon CO2e 0.000411367 kg | standardized CO2e 0.000679299 kg | monitored runtime 74.937 s

=== STANDALONE EFFICIENTNET-LITE0 STAGE 3 EXTERNAL INFERENCE — SEED 112 ===
Validated EfficientNet-Lite0 seed 112: timm.efficientnet_lite0 / unaltered / unaltered / strict=True
Seed 112 completed | external: 3497 images × 2 passes = 6994 processed | energy 0.001468235 kWh | original CodeCarbon CO2e 0.000422335 kg | standardized CO2e 0.000697411 kg | monitored runtime 74.663 s

=== STANDALONE EFFICIENTNET-LITE0 STAGE 3 

## 9. D3 sustainability and accuracy metrics — five-seed mean ± sample SD

In [9]:
results_df=pd.DataFrame(sustainability_rows)
assert len(results_df)==5 and set(results_df.seed)==set(SEEDS)
MEASURES=[
    'total_inference_energy_kWh','codecarbon_emissions_kgCO2e',
    'standardized_CO2e_kg_fixed_factor','monitoring_seconds',
    'energy_kWh_per_processed_image','time_seconds_per_processed_image',
    'throughput_images_per_second','first_pass_accuracy_pct_diagnostic',
    'first_pass_macro_precision_pct','first_pass_macro_recall_pct','first_pass_macro_f1_pct']
summary_rows=[]
for measure in MEASURES:
    values=results_df[measure].astype(float)
    summary_rows.append({'model':'EfficientNet-Lite0','measure':measure,'n_seeds':len(values),
                         'mean':values.mean(),'sample_sd':values.std(ddof=1),
                         'min':values.min(),'max':values.max()})
summary_df=pd.DataFrame(summary_rows)
results_df.to_csv(OUTPUT/'EfficientNet_Lite0_Stage3_Inference_Per_Seed.csv',index=False,float_format='%.12f')
summary_df.to_csv(OUTPUT/'EfficientNet_Lite0_Stage3_Inference_Five_Seed_Mean_SD.csv',index=False,float_format='%.12f')
print(summary_df[['measure','mean','sample_sd']].to_string(index=False,
    formatters={'mean':lambda v:f'{v:.12f}','sample_sd':lambda v:f'{v:.12f}'}))
print('Five-seed inference summary saved.')


                           measure            mean      sample_sd
        total_inference_energy_kWh  0.001444808936 0.000031933651
       codecarbon_emissions_kgCO2e  0.000415596868 0.000009185661
 standardized_CO2e_kg_fixed_factor  0.000686284245 0.000015168484
                monitoring_seconds 75.249015065000 1.044503627447
    energy_kWh_per_processed_image  0.000000206578 0.000000004566
  time_seconds_per_processed_image  0.010759081365 0.000149342812
      throughput_images_per_second 92.958975574597 1.282203538007
first_pass_accuracy_pct_diagnostic 44.003431512725 1.298268244967
    first_pass_macro_precision_pct 46.183458675545 2.780428299654
       first_pass_macro_recall_pct 43.078469573788 1.064987569338
           first_pass_macro_f1_pct 40.472550871753 2.054105659432
Five-seed inference summary saved.


## 10. Historical Stage 2 EfficientNet-Lite0 training-session monitoring — reference only

The following rows and original five-seed means are transcribed from the saved output of uploaded **EfficientNet-Lite0(3).ipynb**. The original training script monitored both training and epoch-wise evaluation, and detected two GPUs; the figures are historical recorded *session* totals, not new Stage 3 measurements. Its reported seed values were rounded to six decimals for energy and CO₂e. Preserve the original historical means at their reported precision; compute SD and standardized carbon estimates from the rounded seed rows and disclose that distinction. All Stage 3 results use the standalone **classification checkpoint**, not a separately trained sustainability checkpoint.

In [10]:
# Historical EfficientNet-Lite0 Stage 2 training-session records transcribed from
# the actual saved final five-seed table in EfficientNet-Lite0(3).ipynb.
# This Stage 2 script included evaluation during training; it did not measure external D2.
historical_rows=[
 {'seed':7,  'stage2_codecarbon_CO2e_kg_reported':0.004057,'training_session_energy_kWh_reported':0.014105,'training_session_time_min_reported':9.600208},
 {'seed':49, 'stage2_codecarbon_CO2e_kg_reported':0.004076,'training_session_energy_kWh_reported':0.014171,'training_session_time_min_reported':9.580558},
 {'seed':112,'stage2_codecarbon_CO2e_kg_reported':0.004103,'training_session_energy_kWh_reported':0.014263,'training_session_time_min_reported':9.649802},
 {'seed':345,'stage2_codecarbon_CO2e_kg_reported':0.004120,'training_session_energy_kWh_reported':0.014323,'training_session_time_min_reported':9.702012},
 {'seed':476,'stage2_codecarbon_CO2e_kg_reported':0.004093,'training_session_energy_kWh_reported':0.014229,'training_session_time_min_reported':9.625188},
]
historical_training_df=pd.DataFrame(historical_rows)
historical_training_df['stage2_standardized_CO2e_kg_fixed_0475_from_rounded_energy']=(
    historical_training_df['training_session_energy_kWh_reported']*CARBON_FACTOR)
historical_training_df['stage2_original_source']='EfficientNet-Lite0(3).ipynb saved final 5-seed output'
historical_training_df['status']='Historical Stage 2 training + epoch-wise test session, not Stage 3 inference'
historical_training_df.to_csv(OUTPUT/'EfficientNet_Lite0_Stage2_Historical_Training_Per_Seed.csv',index=False,float_format='%.12f')
reported_historical_means={
 'stage2_codecarbon_CO2e_kg_reported':0.004089837530504514,
 'training_session_energy_kWh_reported':0.0142181865967307,
 'training_session_time_min_reported':9.631553467114767,
}
historical_summary=[]
for measure in ['stage2_codecarbon_CO2e_kg_reported','training_session_energy_kWh_reported',
                'training_session_time_min_reported','stage2_standardized_CO2e_kg_fixed_0475_from_rounded_energy']:
    vals=historical_training_df[measure].astype(float)
    historical_summary.append({
       'measure':measure,'n_seeds':5,
       'mean_as_reported_stage2_or_from_rounded_data':reported_historical_means.get(measure,float(vals.mean())),
       'mean_from_printed_rounded_seed_rows':float(vals.mean()),
       'sample_sd_from_printed_rounded_seed_rows':float(vals.std(ddof=1)),
       'note':'Source high-precision means retained; SD and standardized CO2e from rounded source seed rows'})
historical_summary_df=pd.DataFrame(historical_summary)
historical_summary_df.to_csv(OUTPUT/'EfficientNet_Lite0_Stage2_Historical_Training_Mean_SD.csv',index=False,float_format='%.12f')

print('\n'+'='*75)
print('HISTORICAL STAGE 2 — EFFICIENTNET-LITE0 TRAINING SESSION (NOT MEASURED IN STAGE 3)')
print('='*75)
print(historical_training_df[['seed','stage2_codecarbon_CO2e_kg_reported',
   'training_session_energy_kWh_reported','stage2_standardized_CO2e_kg_fixed_0475_from_rounded_energy',
   'training_session_time_min_reported']].to_string(index=False,formatters={
     'stage2_codecarbon_CO2e_kg_reported':lambda v:f'{v:.9f}',
     'training_session_energy_kWh_reported':lambda v:f'{v:.9f}',
     'stage2_standardized_CO2e_kg_fixed_0475_from_rounded_energy':lambda v:f'{v:.9f}',
     'training_session_time_min_reported':lambda v:f'{v:.6f}',
 }))
print('Reported mean historical CodeCarbon CO2e (kg):',f"{reported_historical_means['stage2_codecarbon_CO2e_kg_reported']:.12f}")
print('Reported mean historical training-session energy (kWh):',f"{reported_historical_means['training_session_energy_kWh_reported']:.12f}")
print('Reported mean historical training-session time (min):',f"{reported_historical_means['training_session_time_min_reported']:.9f}")
print('Stage 2 standardized CO2e from rounded seed energies (kg):',
      f"{historical_training_df['stage2_standardized_CO2e_kg_fixed_0475_from_rounded_energy'].mean():.12f}")
print('Historical Stage 2 session is distinct from Stage 3 D3 inference; no combined cost claimed.')
print('\n'+'='*75)
print('STAGE 3 — EXTERNAL EFFICIENTNET-LITE0 INFERENCE (NEWLY MONITORED; 3,497 UNIQUE IMAGES)')
print('='*75)
print(results_df[['seed','unique_external_images','number_complete_repetitions','total_processed_images',
   'total_inference_energy_kWh','codecarbon_emissions_kgCO2e',
   'standardized_CO2e_kg_fixed_factor','monitoring_seconds',
   'energy_kWh_per_processed_image','time_seconds_per_processed_image',
   'throughput_images_per_second']].to_string(index=False,formatters={
     'total_inference_energy_kWh':lambda v:f'{v:.9f}',
     'codecarbon_emissions_kgCO2e':lambda v:f'{v:.9f}',
     'standardized_CO2e_kg_fixed_factor':lambda v:f'{v:.9f}',
     'monitoring_seconds':lambda v:f'{v:.3f}',
     'energy_kWh_per_processed_image':lambda v:f'{v:.12f}',
     'time_seconds_per_processed_image':lambda v:f'{v:.9f}',
     'throughput_images_per_second':lambda v:f'{v:.3f}',
 }))
print('CodeCarbon original and 0.475-factor standardized CO2e remain separate.')
print('Five-seed mean standardized CO2e (kg):',f"{results_df['standardized_CO2e_kg_fixed_factor'].mean():.9f}")
print('Five-seed sample SD standardized CO2e (kg):',f"{results_df['standardized_CO2e_kg_fixed_factor'].std(ddof=1):.9f}")



HISTORICAL STAGE 2 — EFFICIENTNET-LITE0 TRAINING SESSION (NOT MEASURED IN STAGE 3)
 seed stage2_codecarbon_CO2e_kg_reported training_session_energy_kWh_reported stage2_standardized_CO2e_kg_fixed_0475_from_rounded_energy training_session_time_min_reported
    7                        0.004057000                          0.014105000                                                0.006699875                           9.600208
   49                        0.004076000                          0.014171000                                                0.006731225                           9.580558
  112                        0.004103000                          0.014263000                                                0.006774925                           9.649802
  345                        0.004120000                          0.014323000                                                0.006803425                           9.702012
  476                        0.004093000                

## 11. Provenance, raw CodeCarbon monitoring logs, and downloadable ZIP

In [11]:
configuration={
 'experiment':'Standalone EfficientNet-Lite0 Stage 3 external inference sustainability',
 'stage2_sustainability_source':'EfficientNet-Lite0(3).ipynb, historical saved notebook outputs',
 'historical_stage2_training':'Historical Stage 2 training and epoch-wise evaluation monitoring; not remeasured or combined with Stage 3 D3 inference',
 'historical_stage2_original_CO2':'Original CodeCarbon results reported in Stage 2 notebook',
 'historical_stage2_standardized_CO2':'Rounded historical energy times 0.475; distinct from original emissions',
 'checkpoint_source':'Frozen Stage 2 standalone EfficientNet-Lite0 classification checkpoints',
 'input_checkpoint_root':str(STANDALONE_CKPT_DIR),
 'not_ensemble_constituent_checkpoints':True,
 'seeds':list(SEEDS),'class_to_index':CLASS_TO_ID,
 'external_dataset':'RealWaste (D3), mounted classification-file dataset',
 'external_unique_images':len(external_eval_df),
 'external_manifest_fingerprint':EXTERNAL_FINGERPRINT,
 'external_manifest_path':str(EXTERNAL_MANIFEST),
 'class_mapping_path':str(CLASS_MAPPING),
 'd3_dataset_path':str(external_download),
 'preprocessing':f'RGB Resize(({INPUT_RESOLUTION},{INPUT_RESOLUTION})) ToTensor Normalize((0.5,)*3,(0.5,)*3)',
 'architecture':'EfficientNetLiteModel wrapper / timm.create_model(efficientnet_lite0, pretrained=False) / six-class classifier',
 'no_training_no_adaptation':True,'classification_checkpoints_unmodified':True,
 'checkpoint_loading':'strict=True; full state_dict keys/shapes and checkpoint seed checks; exact wrapper-prefix adjustment allowed',
 'batch_size':BATCH_SIZE,'minimum_complete_passes':MIN_COMPLETE_PASSES,
 'minimum_monitor_seconds':MIN_MONITOR_SECONDS,
 'codecarbon_sampling_seconds':CODECARBON_SAMPLING_SECONDS,
 'codecarbon_version':installed_version('codecarbon'),
 'device':str(DEVICE),
 'hardware_gpu':torch.cuda.get_device_name(0) if DEVICE.type=='cuda' else None,
 'carbon_factor_fixed_kgCO2e_per_kWh':CARBON_FACTOR,
 'measurement_boundary':'After checkpoint loading and warm-up; full D3 passes, including batch loading/preprocessing; GPU synchronized for timing; no checkpoint load/warm-up in monitored interval.',
 'original_codecarbon_emissions':'Original emissions.csv emissions column, never replaced',
 'standardized_CO2e':'Energy in kWh times fixed carbon-intensity factor; separately displayed',
 'realwaste_d3_image_hashes_verified':True,
 'fair_comparison_note':'Use identical hardware, batch 32, preprocessing and monitoring protocol; confirm classification checkpoint SHA256 registry matches frozen EfficientNet-Lite0 classification run.',
 'created_utc':datetime.now(timezone.utc).isoformat()
}
(OUTPUT/'EfficientNet_Lite0_Stage3_Inference_Configuration.json').write_text(
    json.dumps(configuration,indent=2,ensure_ascii=False),encoding='utf-8')
with zipfile.ZipFile(ZIP_PATH,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for p in sorted(OUTPUT.rglob('*')):
        if p.is_file() and 'original_source_extracted' not in p.parts:
            z.write(p,p.relative_to(OUTPUT))
print('One downloadable archive:',ZIP_PATH)
print('Archive bytes:',ZIP_PATH.stat().st_size)
print('Outputs: standalone EfficientNet-Lite0 Stage 3 inference five-seed CSVs, original CodeCarbon logs, checkpoint registry, historical standalone EfficientNet-Lite0 Stage 2 tables, configuration JSON.')


One downloadable archive: /kaggle/working/EfficientNet_Lite0_D3_Inference_Sustainability_All_Outputs.zip
Archive bytes: 9485
Outputs: standalone EfficientNet-Lite0 Stage 3 inference five-seed CSVs, original CodeCarbon logs, checkpoint registry, historical standalone EfficientNet-Lite0 Stage 2 tables, configuration JSON.
